In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.bbc.com/sport/football/premier-league/table"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

teams, points, goals_for = [], [], []

for row in soup.find_all('tr')[1:21]:
    cols = row.find_all('td')
    if not cols:
        continue
        
    row_data = [c.get_text(separator=" ", strip=True) for c in cols]
    
    numeric_data = [int(val) for val in row_data if val.lstrip('-').isdigit()]
    
    team_candidates = [val for val in row_data if not val.lstrip('-').isdigit() and len(val) > 2]
    
    if len(numeric_data) >= 8 and team_candidates:
        teams.append(team_candidates[0])
        goals_for.append(numeric_data[4])
        points.append(numeric_data[-1]) 


df = pd.DataFrame({'Team': teams, 'Goals Scored': goals_for, 'Points': points})
df.to_csv('premier_league_data.csv', index=False)
print("Data successfully scraped and saved to premier_league_data.csv.")

Data successfully scraped and saved to premier_league_data.csv.


In [2]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px

df = pd.read_csv('premier_league_data.csv')
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Premier League Analytics"),
    dcc.Dropdown(
        id='team-filter', 
        options=[{'label': i, 'value': i} for i in df['Team']], 
        multi=True, 
        placeholder="Select teams to filter..."
    ),
    html.Div(id='kpi-cards', style={'display': 'flex', 'gap': '20px', 'padding': '20px 0'}),
    dcc.Graph(id='points-chart'),
    dcc.Graph(id='goals-chart')
])

@app.callback(
    [Output('kpi-cards', 'children'), Output('points-chart', 'figure'), Output('goals-chart', 'figure')],
    [Input('team-filter', 'value')]
)
def update_dashboard(selected_teams):
    filtered_df = df[df['Team'].isin(selected_teams)] if selected_teams else df
    
    # Generate 4 KPI cards
    kpis = [
        html.Div([html.H3("Total Teams"), html.P(len(filtered_df))]),
        html.Div([html.H3("Avg Points"), html.P(f"{filtered_df['Points'].mean():.1f}")]),
        html.Div([html.H3("Total Goals"), html.P(filtered_df['Goals Scored'].sum())]),
        html.Div([html.H3("Max Points"), html.P(filtered_df['Points'].max())])
    ]
    
    # Generate 2 Charts
    fig_points = px.bar(filtered_df, x='Team', y='Points', title="Team Points Comparison")
    fig_goals = px.bar(filtered_df, x='Team', y='Goals Scored', title="Goals Scored by Team")
    
    return kpis, fig_points, fig_goals

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
